# Module 4.2 — Risk Metrics & Management
## Measuring What Can Destroy You

---

### On the Nature of Financial Risk

Risk, in the colloquial sense, is the possibility of something bad happening. In quantitative finance, we demand more precision: risk is the **probability distribution of losses**, and risk management is the discipline of measuring, monitoring, and controlling exposure to that distribution. This distinction matters because risk is not a single number—it is a shape, a tail, a conditional expectation, a path through time. Different metrics illuminate different aspects of this shape, and a quant who relies on only one measure is like a pilot who flies by altimeter alone.

The 2008 financial crisis exposed a generation of risk models as dangerously incomplete. Value at Risk—the industry's dominant risk measure—told banks they had a 1% chance of losing more than $X$ on any given day. What it didn't tell them was *how much more than $X$* they could lose in that 1% tail. When the tail materialized, the losses were not 1.5$X$ or 2$X$—they were 10$X$, 20$X$, existential. The lesson was not that risk models are useless, but that they must be understood as **conditional statements about probability**, not guarantees. Every risk metric answers a specific question and is silent on all others. The art of risk management is asking enough questions to see the full picture.

This module builds the complete risk measurement toolkit that every quant needs: from Value at Risk and its superior cousin Expected Shortfall, through drawdown analysis and performance attribution, to risk budgeting and the construction of risk parity portfolios. At each stage, we ask not only *how* to compute the metric but *why a quant needs it* and *where it fails*.

---

### Learning Objectives

By the end of this module, you will:

1. **Compute** Value at Risk using parametric, historical, and Monte Carlo methods
2. **Understand** why Expected Shortfall (CVaR) is superior to VaR and implement it
3. **Analyze** maximum drawdown, drawdown duration, and recovery dynamics
4. **Decompose** portfolio returns into alpha, beta, and factor contributions
5. **Implement** risk budgeting and equal risk contribution (risk parity) portfolios
6. **Build** a comprehensive risk monitoring dashboard
7. **Recognize** the limitations of every metric and the dangers of model overconfidence

---

### Module Contents

1. **Value at Risk (VaR)** — The question that started an industry
2. **Expected Shortfall (CVaR)** — What VaR refuses to tell you
3. **Maximum Drawdown** — The metric that captures lived experience
4. **Performance Attribution** — Where did the returns come from?
5. **Risk Budgeting & Risk Parity** — Allocating risk, not capital
6. **Risk Monitoring Dashboard** — Putting it all together

---

*"Risk comes from not knowing what you're doing." — Warren Buffett*

*Risk management comes from knowing what you don't know.*

In [ ]:
# ========================================
# Initial Setup and Configuration
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

PLOT_CONFIG = {
    'figure.figsize': (14, 7),
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.dpi': 100,
    'lines.linewidth': 2,
}
plt.rcParams.update(PLOT_CONFIG)

COLORS = {
    'var': '#C62828',
    'cvar': '#880E4F',
    'return': '#2E7D32',
    'drawdown': '#B71C1C',
    'portfolio': '#1565C0',
    'benchmark': '#F57C00',
    'risk_budget': '#6A1B9A',
    'neutral': '#455A64',
    'confidence': '#90CAF9',
    'tail': '#D50000',
}

print("=" * 80)
print(" " * 16 + "MODULE 4.2: RISK METRICS & MANAGEMENT")
print("=" * 80)
print(f"✓ Environment configured successfully")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("\n📚 Ready to measure what can destroy you!\n")


# ========================================
# Shared Data Generation
# ========================================

def generate_portfolio_data(n_days: int = 1260, seed: int = 42) -> Dict:
    """Generate a multi-asset portfolio with realistic properties including
    volatility clustering, fat tails, and a stress episode."""
    rng = np.random.RandomState(seed)
    names = ['US Equity', 'Intl Equity', 'US Bonds', 'Corp Bonds',
             'REITs', 'Commodities', 'Gold']
    n = len(names)

    ann_mu = np.array([0.10, 0.08, 0.03, 0.05, 0.09, 0.06, 0.04])
    ann_vol = np.array([0.16, 0.18, 0.04, 0.06, 0.20, 0.18, 0.15])

    corr = np.array([
        [1.00, 0.75, -0.10, 0.15, 0.60, 0.20, 0.00],
        [0.75, 1.00, -0.05, 0.10, 0.55, 0.25, 0.05],
        [-0.10, -0.05, 1.00, 0.80, 0.10, -0.05, 0.30],
        [0.15, 0.10, 0.80, 1.00, 0.20, 0.00, 0.15],
        [0.60, 0.55, 0.10, 0.20, 1.00, 0.25, 0.05],
        [0.20, 0.25, -0.05, 0.00, 0.25, 1.00, 0.35],
        [0.00, 0.05, 0.30, 0.15, 0.05, 0.35, 1.00],
    ])

    cov = np.outer(ann_vol, ann_vol) * corr
    L = np.linalg.cholesky(cov / 252)

    # Generate with volatility clustering and a stress event
    vol_scale = np.ones(n_days)
    # Stress period: days 600-650 (a market crash)
    vol_scale[600:650] = 3.0
    vol_scale[650:700] = 2.0
    # Some general clustering
    for t in range(1, n_days):
        vol_scale[t] = max(0.5, 0.95 * vol_scale[t] + 0.05 * vol_scale[t-1])

    daily_returns = np.zeros((n_days, n))
    for t in range(n_days):
        Z = rng.standard_t(df=5, size=n)  # fat tails via t-distribution
        Z = Z / np.sqrt(5/3)  # scale to unit variance
        daily_returns[t] = ann_mu / 252 + vol_scale[t] * (L @ Z)

    # Make the crash negative
    daily_returns[600:640, [0,1,4]] -= 0.015
    daily_returns[600:640, [2,6]] += 0.003  # flight to quality

    dates = pd.bdate_range(start='2020-01-02', periods=n_days)
    returns_df = pd.DataFrame(daily_returns, index=dates, columns=names)

    # Portfolio weights (60/40 tilted)
    weights = np.array([0.30, 0.15, 0.20, 0.10, 0.10, 0.08, 0.07])

    port_returns = returns_df.values @ weights
    port_series = pd.Series(port_returns, index=dates, name='Portfolio')

    # Benchmark (simple 60/40)
    bench_w = np.array([0.60, 0.0, 0.40, 0.0, 0.0, 0.0, 0.0])
    bench_returns = returns_df.values @ bench_w
    bench_series = pd.Series(bench_returns, index=dates, name='Benchmark')

    return {
        'returns': returns_df,
        'portfolio': port_series,
        'benchmark': bench_series,
        'weights': pd.Series(weights, index=names),
        'names': names,
        'cov': pd.DataFrame(cov, index=names, columns=names),
    }


data = generate_portfolio_data()
port = data['portfolio']
bench = data['benchmark']

print(f"Data period: {port.index[0].strftime('%Y-%m-%d')} to {port.index[-1].strftime('%Y-%m-%d')} ({len(port)} days)")
print(f"Portfolio Ann. Return: {port.mean()*252:.2%}  |  Vol: {port.std()*np.sqrt(252):.2%}")
print(f"Benchmark Ann. Return: {bench.mean()*252:.2%}  |  Vol: {bench.std()*np.sqrt(252):.2%}")

---

## 1. Value at Risk (VaR)

### Why Quants Need This

VaR is the **lingua franca of risk communication**. When a portfolio manager tells the risk committee "our 1-day 99% VaR is $2.5 million," everyone in the room understands: there is a 1% chance of losing more than $2.5M tomorrow. Regulators (Basel III) mandate VaR calculations for capital requirements. Prime brokers use VaR to set margin. Risk limits are expressed in VaR terms. Whether or not you think VaR is the best risk measure, you will encounter it every day of your career as a quant. You must know how to compute it, how to interpret it, and—critically—where it lies to you.

### 1.1 Definition

The **Value at Risk** at confidence level $\alpha$ (e.g., 99%) over horizon $h$ is the loss level $\ell$ such that:

$$\mathbb{P}(L > \text{VaR}_\alpha) = 1 - \alpha$$

Equivalently, $\text{VaR}_\alpha$ is the $\alpha$-quantile of the loss distribution (or the negative of the $(1-\alpha)$-quantile of the return distribution).

### 1.2 Three Methods

**Parametric (Variance-Covariance)**: Assumes returns are normally distributed. VaR is then:

$$\text{VaR}_\alpha = -(\mu + z_\alpha \cdot \sigma)$$

where $z_\alpha = \Phi^{-1}(1-\alpha)$, $\mu$ is the expected return, and $\sigma$ is the portfolio volatility. Fast and simple, but wrong whenever returns have fat tails (which they always do).

**Historical Simulation**: Uses the actual empirical distribution of past returns. Sort the last $T$ returns, take the $(1-\alpha)$-quantile. No distributional assumption, but assumes the past is representative of the future.

**Monte Carlo**: Simulates thousands of possible return scenarios from a fitted model (e.g., multivariate normal, or copula + marginals). Takes the $(1-\alpha)$-quantile of simulated losses. The most flexible method—can handle non-linear positions, complex dependencies, and fat tails—but the most computationally expensive.

### 1.3 Where VaR Fails

VaR has two critical flaws:
1. It says nothing about the **severity** of losses beyond the VaR level. A portfolio with VaR of $1M could lose $1.1M or $100M in the tail—VaR cannot distinguish between these.
2. VaR is **not subadditive**: the VaR of a combined portfolio can exceed the sum of individual VaRs, violating the principle that diversification should reduce risk. This makes it unsuitable as a risk measure in the axiomatic sense of Artzner et al. (1999).

In [ ]:
# ========================================
# Value at Risk: Three Methods
# ========================================

class VaRCalculator:
    """Comprehensive Value at Risk calculator implementing parametric,
    historical, and Monte Carlo methods."""

    def __init__(self, returns: pd.Series, portfolio_value: float = 1_000_000):
        self.returns = returns.dropna()
        self.pv = portfolio_value
        self.mu = returns.mean()
        self.sigma = returns.std()

    def parametric(self, confidence: float = 0.99, horizon: int = 1) -> Dict:
        """Parametric (Gaussian) VaR."""
        z = stats.norm.ppf(1 - confidence)
        var_pct = -(self.mu * horizon + z * self.sigma * np.sqrt(horizon))
        return {'method': 'Parametric', 'VaR_pct': var_pct,
                'VaR_dollar': var_pct * self.pv, 'confidence': confidence}

    def historical(self, confidence: float = 0.99, horizon: int = 1) -> Dict:
        """Historical simulation VaR."""
        if horizon > 1:
            # Use overlapping multi-day returns
            multi_returns = self.returns.rolling(horizon).sum().dropna()
        else:
            multi_returns = self.returns
        var_pct = -np.percentile(multi_returns, (1 - confidence) * 100)
        return {'method': 'Historical', 'VaR_pct': var_pct,
                'VaR_dollar': var_pct * self.pv, 'confidence': confidence}

    def monte_carlo(self, confidence: float = 0.99, horizon: int = 1,
                     n_sims: int = 50000, seed: int = 42) -> Dict:
        """Monte Carlo VaR using fitted t-distribution for fat tails."""
        rng = np.random.RandomState(seed)
        # Fit Student-t distribution
        params = stats.t.fit(self.returns)
        df_t, loc_t, scale_t = params

        sim_returns = stats.t.rvs(df_t, loc=loc_t * horizon,
                                   scale=scale_t * np.sqrt(horizon),
                                   size=n_sims, random_state=rng)
        var_pct = -np.percentile(sim_returns, (1 - confidence) * 100)
        return {'method': f'Monte Carlo (t, df={df_t:.1f})', 'VaR_pct': var_pct,
                'VaR_dollar': var_pct * self.pv, 'confidence': confidence,
                'sim_returns': sim_returns}

    def all_methods(self, confidence: float = 0.99, horizon: int = 1) -> pd.DataFrame:
        """Run all three methods and return comparison."""
        results = [
            self.parametric(confidence, horizon),
            self.historical(confidence, horizon),
            self.monte_carlo(confidence, horizon),
        ]
        df = pd.DataFrame([{k: v for k, v in r.items() if k != 'sim_returns'}
                           for r in results])
        return df


# --- Compute VaR ---

calc = VaRCalculator(port, portfolio_value=10_000_000)

print("=" * 80)
print("VALUE AT RISK — THREE METHODS")
print("=" * 80)

for conf in [0.95, 0.99]:
    print(f"\n--- {conf:.0%} Confidence, 1-Day Horizon, $10M Portfolio ---")
    var_table = calc.all_methods(confidence=conf)
    print(f"\n{'Method':<28} {'VaR (%)':>10} {'VaR ($)':>14}")
    print("-" * 56)
    for _, row in var_table.iterrows():
        print(f"{row['method']:<28} {row['VaR_pct']:>9.4%} {row['VaR_dollar']:>13,.0f}")

# --- VaR backtest: how often are breaches? ---

print(f"\n{'=' * 80}")
print("VaR BACKTEST (99% confidence)")
print(f"{'=' * 80}")

rolling_window = 252
var_series_param = []
var_series_hist = []

for i in range(rolling_window, len(port)):
    window = port.iloc[i-rolling_window:i]
    mu_w = window.mean()
    sigma_w = window.std()
    z = stats.norm.ppf(0.01)
    var_series_param.append(-(mu_w + z * sigma_w))
    var_series_hist.append(-np.percentile(window, 1))

var_param = pd.Series(var_series_param, index=port.index[rolling_window:])
var_hist = pd.Series(var_series_hist, index=port.index[rolling_window:])
actual = port.iloc[rolling_window:]

breaches_param = (actual < -var_param).sum()
breaches_hist = (actual < -var_hist).sum()
expected_breaches = len(actual) * 0.01

print(f"  Expected breaches (1%): {expected_breaches:.1f}")
print(f"  Parametric breaches:    {breaches_param} ({breaches_param/len(actual):.2%})")
print(f"  Historical breaches:    {breaches_hist} ({breaches_hist/len(actual):.2%})")

# --- Visualization ---

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Value at Risk: Measuring the Boundary of Normal Losses",
             fontsize=18, fontweight='bold', y=1.02)

# Panel 1: Return distribution with VaR lines
ax = axes[0, 0]
ax.hist(port * 100, bins=80, density=True, color=COLORS['return'], alpha=0.6, edgecolor='white')
var_99_param = calc.parametric(0.99)['VaR_pct']
var_99_hist = calc.historical(0.99)['VaR_pct']
mc_result = calc.monte_carlo(0.99)
var_99_mc = mc_result['VaR_pct']
ax.axvline(-var_99_param * 100, color='blue', linewidth=2, linestyle='--', label=f'Parametric: {var_99_param:.2%}')
ax.axvline(-var_99_hist * 100, color='red', linewidth=2, linestyle='-.', label=f'Historical: {var_99_hist:.2%}')
ax.axvline(-var_99_mc * 100, color='purple', linewidth=2, linestyle=':', label=f'Monte Carlo: {var_99_mc:.2%}')
# Shade the tail
tail_returns = port[port < -var_99_hist] * 100
if len(tail_returns) > 0:
    ax.hist(tail_returns, bins=20, density=True, color=COLORS['tail'], alpha=0.8, edgecolor='white')
ax.set_title('Return Distribution with 99% VaR', fontsize=14)
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

# Panel 2: VaR backtest
ax = axes[0, 1]
ax.plot(actual.index, actual * 100, color=COLORS['return'], linewidth=0.5, alpha=0.6, label='Returns')
ax.plot(var_param.index, -var_param * 100, color='blue', linewidth=1, label='Parametric VaR')
ax.plot(var_hist.index, -var_hist * 100, color='red', linewidth=1, linestyle='--', label='Historical VaR')
breach_mask = actual < -var_hist
ax.scatter(actual.index[breach_mask], actual[breach_mask] * 100,
           color=COLORS['tail'], s=30, zorder=5, label='Breaches')
ax.set_title('VaR Backtest: Breaches Over Time', fontsize=14)
ax.set_ylabel('Return (%)')
ax.legend(fontsize=9, loc='lower left')

# Panel 3: Monte Carlo simulation
ax = axes[1, 0]
sim_rets = mc_result['sim_returns'] * 100
ax.hist(sim_rets, bins=100, density=True, color=COLORS['neutral'], alpha=0.5, edgecolor='white')
ax.axvline(-var_99_mc * 100, color=COLORS['var'], linewidth=2, label=f'99% VaR: {var_99_mc:.2%}')
x_norm = np.linspace(sim_rets.min(), sim_rets.max(), 300)
ax.plot(x_norm, stats.norm.pdf(x_norm, port.mean()*100, port.std()*100),
        color='blue', linewidth=1.5, linestyle='--', label='Normal Fit', alpha=0.7)
ax.set_title('Monte Carlo Simulated Returns (t-distribution)', fontsize=14)
ax.set_xlabel('Return (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

# Panel 4: VaR at different confidence levels
ax = axes[1, 1]
conf_levels = np.linspace(0.90, 0.999, 50)
var_by_conf = {}
for method_name, method in [('Parametric', calc.parametric), ('Historical', calc.historical)]:
    var_by_conf[method_name] = [method(c)['VaR_pct'] * 100 for c in conf_levels]

ax.plot(conf_levels * 100, var_by_conf['Parametric'], color='blue', linewidth=2, label='Parametric')
ax.plot(conf_levels * 100, var_by_conf['Historical'], color='red', linewidth=2, label='Historical')
ax.set_title('VaR Across Confidence Levels', fontsize=14)
ax.set_xlabel('Confidence Level (%)')
ax.set_ylabel('VaR (%)')
ax.legend()

plt.tight_layout()
plt.show()

---

## 2. Expected Shortfall (CVaR): What VaR Refuses to Tell You

### Why Quants Need This

After the financial crisis, regulators (Basel III → Basel IV) began shifting from VaR to **Expected Shortfall** as the primary regulatory risk measure. The reason is simple: VaR tells you the threshold of the tail, but Expected Shortfall tells you the **average loss in the tail**. If your strategy occasionally loses 1% but once in a crisis loses 30%, VaR and ES tell very different stories. ES is also a **coherent risk measure** (satisfying monotonicity, translation invariance, positive homogeneity, and—crucially—subadditivity), which means it properly rewards diversification. Every modern risk system should compute both.

### 2.1 Definition

The **Expected Shortfall** (also called Conditional VaR or CVaR) at level $\alpha$ is:

$$\text{ES}_\alpha = -\mathbb{E}[R \mid R < -\text{VaR}_\alpha] = \frac{1}{1-\alpha} \int_0^{1-\alpha} \text{VaR}_u \, du$$

For a normal distribution, ES has a closed form:

$$\text{ES}_\alpha = -\mu + \sigma \cdot \frac{\phi(z_\alpha)}{1 - \alpha}$$

where $\phi$ is the standard normal PDF and $z_\alpha = \Phi^{-1}(1-\alpha)$.

### 2.2 The Coherence Argument

Artzner et al. (1999) defined four axioms that a "good" risk measure should satisfy. VaR fails subadditivity: $\text{VaR}(A + B) > \text{VaR}(A) + \text{VaR}(B)$ can occur, meaning diversification appears to *increase* risk. ES satisfies all four axioms, making it theoretically superior. In practice, the difference matters most for portfolios with non-linear risk profiles (options, credit derivatives).

In [ ]:
# ========================================
# Expected Shortfall / CVaR
# ========================================

class ESCalculator:
    """Expected Shortfall calculator with parametric, historical, and MC methods."""

    def __init__(self, returns: pd.Series, portfolio_value: float = 1_000_000):
        self.returns = returns.dropna()
        self.pv = portfolio_value
        self.mu = returns.mean()
        self.sigma = returns.std()

    def parametric(self, confidence: float = 0.99) -> Dict:
        z = stats.norm.ppf(1 - confidence)
        es_pct = -(self.mu - self.sigma * stats.norm.pdf(z) / (1 - confidence))
        return {'method': 'Parametric', 'ES_pct': es_pct, 'ES_dollar': es_pct * self.pv}

    def historical(self, confidence: float = 0.99) -> Dict:
        threshold = np.percentile(self.returns, (1 - confidence) * 100)
        tail_returns = self.returns[self.returns <= threshold]
        es_pct = -tail_returns.mean()
        return {'method': 'Historical', 'ES_pct': es_pct, 'ES_dollar': es_pct * self.pv,
                'n_tail_obs': len(tail_returns)}

    def monte_carlo(self, confidence: float = 0.99, n_sims: int = 100000,
                     seed: int = 42) -> Dict:
        rng = np.random.RandomState(seed)
        params = stats.t.fit(self.returns)
        sim = stats.t.rvs(*params, size=n_sims, random_state=rng)
        threshold = np.percentile(sim, (1 - confidence) * 100)
        es_pct = -sim[sim <= threshold].mean()
        return {'method': 'Monte Carlo', 'ES_pct': es_pct, 'ES_dollar': es_pct * self.pv}


es_calc = ESCalculator(port, portfolio_value=10_000_000)
var_calc = VaRCalculator(port, portfolio_value=10_000_000)

print("=" * 80)
print("VaR vs EXPECTED SHORTFALL COMPARISON")
print("=" * 80)

print(f"\nPortfolio: $10,000,000 | 1-Day Horizon")
print(f"\n{'Method':<16} {'Conf':>6} {'VaR (%)':>10} {'ES (%)':>10} {'ES/VaR':>8} {'VaR ($)':>12} {'ES ($)':>12}")
print("-" * 78)

for conf in [0.95, 0.99]:
    for method in ['parametric', 'historical', 'monte_carlo']:
        var_r = getattr(var_calc, method)(conf)
        es_r = getattr(es_calc, method)(conf)
        ratio = es_r['ES_pct'] / var_r['VaR_pct'] if var_r['VaR_pct'] > 0 else 0
        name = method.replace('_', ' ').title()
        print(f"{name:<16} {conf:>5.0%} {var_r['VaR_pct']:>9.4%} {es_r['ES_pct']:>9.4%} "
              f"{ratio:>8.2f} {var_r['VaR_dollar']:>11,.0f} {es_r['ES_dollar']:>11,.0f}")

print(f"\n💡 ES is always larger than VaR (it includes the average loss beyond VaR).")
print(f"   The ES/VaR ratio reveals tail heaviness: normal ≈ 1.14, fat tails > 1.3.")

# --- Visualization ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Expected Shortfall: Looking Inside the Tail",
             fontsize=18, fontweight='bold')

# Panel 1: VaR vs ES on the distribution
ax = axes[0]
sorted_rets = np.sort(port.values) * 100
var_99 = calc.historical(0.99)['VaR_pct']
es_99 = es_calc.historical(0.99)['ES_pct']

ax.hist(sorted_rets, bins=80, density=True, color=COLORS['return'], alpha=0.5, edgecolor='white')
tail_mask = sorted_rets < -var_99 * 100
ax.hist(sorted_rets[tail_mask], bins=20, density=True, color=COLORS['tail'], alpha=0.8, label='Tail (1%)')
ax.axvline(-var_99 * 100, color=COLORS['var'], linewidth=2.5, label=f'VaR₉₉: {var_99:.2%}')
ax.axvline(-es_99 * 100, color=COLORS['cvar'], linewidth=2.5, linestyle='--', label=f'ES₉₉: {es_99:.2%}')
ax.set_title('VaR Marks the Boundary; ES Reveals the Interior', fontsize=14)
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)

# Panel 2: Rolling VaR and ES
ax = axes[1]
roll_w = 252
rolling_var = port.rolling(roll_w).apply(lambda x: -np.percentile(x, 1), raw=True)
rolling_es = port.rolling(roll_w).apply(lambda x: -x[x <= np.percentile(x, 1)].mean(), raw=True)
ax.plot(rolling_var.index, rolling_var * 100, color=COLORS['var'], linewidth=1.5, label='Rolling VaR (99%)')
ax.plot(rolling_es.index, rolling_es * 100, color=COLORS['cvar'], linewidth=1.5, label='Rolling ES (99%)')
ax.fill_between(rolling_es.index, rolling_var * 100, rolling_es * 100,
                alpha=0.2, color=COLORS['cvar'], label='ES − VaR gap')
ax.set_title('Rolling VaR and ES (252-Day Window)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Risk (%)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---

## 3. Maximum Drawdown: The Metric That Captures Lived Experience

### Why Quants Need This

No risk metric captures the **psychological reality** of managing money better than drawdown. VaR is a one-day concept; Sharpe ratios are long-run averages. But when your portfolio drops 25% from its peak and stays underwater for 14 months, the experience is visceral—for you, for your investors, and for your career. **Maximum drawdown (MDD)** is the largest peak-to-trough decline in portfolio value. **Drawdown duration** is how long you stay underwater. Together, they answer the question every allocator really cares about: "How bad can it get, and how long will it last?"

### 3.1 Definition

Given a cumulative return series $V_t$, the drawdown at time $t$ is:

$$\text{DD}_t = \frac{V_t - \max_{s \leq t} V_s}{\max_{s \leq t} V_s}$$

The maximum drawdown is $\text{MDD} = \min_t \text{DD}_t$.

The **Calmar ratio** (annualized return / MDD) and **Sterling ratio** (annualized return / average drawdown) normalize performance by drawdown risk.

### 3.2 Drawdown Duration

A drawdown period begins when the portfolio falls below its high-water mark and ends when a new high is established. The **maximum drawdown duration** is the longest such period. For funds with annual redemption windows, a drawdown duration exceeding 12 months can trigger mass redemptions—a liquidity crisis that compounds the drawdown.

In [ ]:
# ========================================
# Drawdown Analysis
# ========================================

@dataclass
class DrawdownResult:
    """Complete drawdown analysis output."""
    drawdown_series: pd.Series
    max_drawdown: float
    max_dd_start: datetime
    max_dd_trough: datetime
    max_dd_end: Optional[datetime]
    max_dd_duration_days: int
    avg_drawdown: float
    calmar_ratio: float
    top_drawdowns: pd.DataFrame


def analyze_drawdowns(returns: pd.Series, top_n: int = 5) -> DrawdownResult:
    """Comprehensive drawdown analysis."""
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max

    # Find top N drawdown periods
    is_underwater = drawdown < 0
    dd_periods = []
    in_dd = False
    start = None

    for i, (date, dd) in enumerate(drawdown.items()):
        if dd < 0 and not in_dd:
            start = date
            in_dd = True
        elif dd >= 0 and in_dd:
            end = date
            period = drawdown[start:end]
            trough_date = period.idxmin()
            dd_periods.append({
                'Start': start,
                'Trough': trough_date,
                'Recovery': end,
                'Depth': period.min(),
                'Duration (days)': (end - start).days,
                'To Trough (days)': (trough_date - start).days,
            })
            in_dd = False

    if in_dd:
        period = drawdown[start:]
        trough_date = period.idxmin()
        dd_periods.append({
            'Start': start, 'Trough': trough_date, 'Recovery': None,
            'Depth': period.min(),
            'Duration (days)': (drawdown.index[-1] - start).days,
            'To Trough (days)': (trough_date - start).days,
        })

    dd_df = pd.DataFrame(dd_periods).sort_values('Depth').head(top_n).reset_index(drop=True)

    mdd = drawdown.min()
    mdd_trough = drawdown.idxmin()
    ann_ret = returns.mean() * 252
    calmar = ann_ret / abs(mdd) if mdd != 0 else 0

    # Find MDD start
    peak_before_trough = running_max[:mdd_trough].idxmax()

    return DrawdownResult(
        drawdown_series=drawdown,
        max_drawdown=mdd,
        max_dd_start=peak_before_trough,
        max_dd_trough=mdd_trough,
        max_dd_end=dd_df.iloc[0]['Recovery'] if len(dd_df) > 0 else None,
        max_dd_duration_days=dd_df.iloc[0]['Duration (days)'] if len(dd_df) > 0 else 0,
        avg_drawdown=drawdown[drawdown < 0].mean() if (drawdown < 0).any() else 0,
        calmar_ratio=calmar,
        top_drawdowns=dd_df,
    )


# --- Run analysis ---

dd_port = analyze_drawdowns(port)
dd_bench = analyze_drawdowns(bench)

print("=" * 80)
print("DRAWDOWN ANALYSIS")
print("=" * 80)

print(f"\n{'Metric':<28} {'Portfolio':>14} {'Benchmark':>14}")
print("-" * 58)
print(f"{'Max Drawdown:':<28} {dd_port.max_drawdown:>13.2%} {dd_bench.max_drawdown:>13.2%}")
print(f"{'MDD Duration (days):':<28} {dd_port.max_dd_duration_days:>14} {dd_bench.max_dd_duration_days:>14}")
print(f"{'Average Drawdown:':<28} {dd_port.avg_drawdown:>13.2%} {dd_bench.avg_drawdown:>13.2%}")
print(f"{'Calmar Ratio:':<28} {dd_port.calmar_ratio:>14.3f} {dd_bench.calmar_ratio:>14.3f}")

print(f"\nTop 5 Drawdown Periods (Portfolio):")
for _, row in dd_port.top_drawdowns.iterrows():
    rec = row['Recovery'].strftime('%Y-%m-%d') if row['Recovery'] is not None else 'Ongoing'
    print(f"  {row['Start'].strftime('%Y-%m-%d')} → {row['Trough'].strftime('%Y-%m-%d')} → {rec}  "
          f"Depth: {row['Depth']:.2%}  Duration: {row['Duration (days)']}d")

# --- Visualization ---

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)
fig.suptitle("Drawdown Analysis: The Path of Pain",
             fontsize=18, fontweight='bold', y=1.01)

# Panel 1: Cumulative returns
ax = axes[0]
cum_port = (1 + port).cumprod()
cum_bench = (1 + bench).cumprod()
ax.plot(cum_port.index, cum_port, color=COLORS['portfolio'], linewidth=1.5, label='Portfolio')
ax.plot(cum_bench.index, cum_bench, color=COLORS['benchmark'], linewidth=1.5, label='Benchmark')
ax.plot(cum_port.index, cum_port.cummax(), color=COLORS['portfolio'], linewidth=0.8,
        linestyle=':', alpha=0.5, label='High-Water Mark')
ax.set_title('Cumulative Return and High-Water Mark', fontsize=14)
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=9)

# Panel 2: Drawdown series
ax = axes[1]
ax.fill_between(dd_port.drawdown_series.index, dd_port.drawdown_series * 100, 0,
                color=COLORS['drawdown'], alpha=0.4, label='Portfolio DD')
ax.plot(dd_bench.drawdown_series.index, dd_bench.drawdown_series * 100,
        color=COLORS['benchmark'], linewidth=1, label='Benchmark DD')
ax.axhline(dd_port.max_drawdown * 100, color='black', linewidth=0.8, linestyle='--',
           label=f'MDD: {dd_port.max_drawdown:.1%}')
ax.set_title('Underwater Equity Curve', fontsize=14)
ax.set_ylabel('Drawdown (%)')
ax.legend(fontsize=9)

# Panel 3: Drawdown duration histogram
ax = axes[2]
durations = dd_port.top_drawdowns['Duration (days)'].values
all_dd_depths = dd_port.top_drawdowns['Depth'].values * 100
colors_dd = plt.cm.Reds(np.linspace(0.4, 0.9, len(durations)))
bars = ax.barh(range(len(durations)), durations, color=colors_dd, edgecolor='white', height=0.6)
for i, (dur, depth) in enumerate(zip(durations, all_dd_depths)):
    ax.text(dur + 5, i, f'{depth:.1f}% depth', va='center', fontsize=10)
ax.set_yticks(range(len(durations)))
ax.set_yticklabels([f'DD #{i+1}' for i in range(len(durations))])
ax.set_title('Top Drawdown Periods: Duration and Depth', fontsize=14)
ax.set_xlabel('Duration (calendar days)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

---

## 4. Performance Attribution & Risk Budgeting

### Why Quants Need This

**Performance attribution** answers the most important question in portfolio management: *where did the returns come from?* Was the portfolio's outperformance driven by smart sector bets (allocation effect), or by picking the best stocks within each sector (selection effect), or simply by loading on the market factor? Without attribution, you cannot distinguish skill from luck, and you cannot replicate what worked or fix what didn't.

**Risk budgeting** flips the attribution lens from returns to risk. Instead of asking "how much capital is in each asset," it asks "how much risk is each asset contributing?" This leads to **risk parity**—the principle that each asset should contribute equally to total portfolio risk, a framework that has powered some of the most successful institutional portfolios (Bridgewater's All Weather, AQR's risk parity strategies).

### 4.1 Performance Decomposition

Active return relative to a benchmark decomposes into:

$$R_p - R_b = \underbrace{\sum_i (w_i^p - w_i^b) r_i^b}_{\text{Allocation}} + \underbrace{\sum_i w_i^b (r_i^p - r_i^b)}_{\text{Selection}} + \underbrace{\sum_i (w_i^p - w_i^b)(r_i^p - r_i^b)}_{\text{Interaction}}$$

### 4.2 Marginal Risk Contribution

The **marginal contribution to risk (MCR)** of asset $i$ is:

$$\text{MCR}_i = w_i \cdot \frac{\partial \sigma_p}{\partial w_i} = w_i \cdot \frac{(\boldsymbol{\Sigma} \mathbf{w})_i}{\sigma_p}$$

The sum of all MCRs equals total portfolio volatility: $\sum_i \text{MCR}_i = \sigma_p$.

A **risk parity** portfolio sets all MCRs equal: $\text{MCR}_1 = \text{MCR}_2 = \cdots = \text{MCR}_N$, so each asset contributes $\sigma_p / N$ to total risk.

In [ ]:
# ========================================
# Performance Attribution & Risk Budgeting
# ========================================

class RiskBudget:
    """Risk contribution analysis and risk parity optimization."""

    def __init__(self, returns: pd.DataFrame, weights: np.ndarray):
        self.returns = returns
        self.cov = returns.cov() * 252
        self.weights = weights
        self.names = list(returns.columns)

    def portfolio_vol(self, w: np.ndarray = None) -> float:
        w = w if w is not None else self.weights
        return np.sqrt(w @ self.cov.values @ w)

    def marginal_contribution(self, w: np.ndarray = None) -> np.ndarray:
        w = w if w is not None else self.weights
        sigma = self.portfolio_vol(w)
        mcr = w * (self.cov.values @ w) / sigma
        return mcr

    def percentage_contribution(self, w: np.ndarray = None) -> np.ndarray:
        mcr = self.marginal_contribution(w)
        return mcr / mcr.sum()

    def risk_parity_weights(self) -> np.ndarray:
        """Find weights where each asset contributes equally to portfolio risk."""
        n = len(self.weights)
        target_risk = 1.0 / n

        def objective(w):
            sigma = np.sqrt(w @ self.cov.values @ w)
            mcr = w * (self.cov.values @ w) / sigma
            pct_contrib = mcr / mcr.sum()
            return np.sum((pct_contrib - target_risk) ** 2)

        result = minimize(
            objective,
            x0=np.ones(n) / n,
            method='SLSQP',
            bounds=[(0.01, 0.5)] * n,
            constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        )
        return result.x


# --- Risk contribution analysis ---

rb = RiskBudget(data['returns'], data['weights'].values)

mcr = rb.marginal_contribution()
pct_contrib = rb.percentage_contribution()
rp_weights = rb.risk_parity_weights()
rp_pct_contrib = rb.percentage_contribution(rp_weights)

print("=" * 80)
print("RISK CONTRIBUTION ANALYSIS")
print("=" * 80)
print(f"\nPortfolio Volatility: {rb.portfolio_vol():.2%}")

print(f"\n{'Asset':<16} {'Weight':>8} {'MCR':>10} {'% Risk':>10} {'RP Wt':>8} {'RP % Risk':>10}")
print("-" * 66)
for i, name in enumerate(rb.names):
    print(f"{name:<16} {data['weights'].values[i]:>7.1%} {mcr[i]:>9.4%} {pct_contrib[i]:>9.1%} "
          f"{rp_weights[i]:>7.1%} {rp_pct_contrib[i]:>9.1%}")

print(f"\nRisk Parity Vol: {rb.portfolio_vol(rp_weights):.2%}")
print(f"\n💡 In the original portfolio, US Equity contributes {pct_contrib[0]:.0%} of risk")
print(f"   despite being only {data['weights'].values[0]:.0%} of capital.")
print(f"   Risk parity rebalances so each asset contributes ≈{1/len(rb.names):.0%} of risk.")

# --- Performance metrics summary ---

print(f"\n{'=' * 80}")
print("COMPREHENSIVE PERFORMANCE METRICS")
print(f"{'=' * 80}")

def compute_all_metrics(returns: pd.Series, benchmark: pd.Series,
                         rf: float = 0.04) -> Dict:
    """Every metric a quant needs, in one place."""
    rf_daily = rf / 252
    excess = returns - rf_daily
    active = returns - benchmark

    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    cum = (1 + returns).cumprod()
    dd = (cum / cum.cummax() - 1)

    downside = returns[returns < 0]
    downside_vol = downside.std() * np.sqrt(252) if len(downside) > 0 else 0

    return {
        'Ann. Return': f"{ann_ret:.2%}",
        'Ann. Volatility': f"{ann_vol:.2%}",
        'Sharpe Ratio': f"{(ann_ret - rf) / ann_vol:.3f}" if ann_vol > 0 else 'N/A',
        'Sortino Ratio': f"{(ann_ret - rf) / downside_vol:.3f}" if downside_vol > 0 else 'N/A',
        'Calmar Ratio': f"{ann_ret / abs(dd.min()):.3f}" if dd.min() != 0 else 'N/A',
        'Max Drawdown': f"{dd.min():.2%}",
        'Skewness': f"{stats.skew(returns):.3f}",
        'Kurtosis': f"{stats.kurtosis(returns):.3f}",
        'VaR (99%)': f"{-np.percentile(returns, 1):.4%}",
        'ES (99%)': f"{-returns[returns <= np.percentile(returns, 1)].mean():.4%}",
        'Tracking Error': f"{active.std() * np.sqrt(252):.2%}",
        'Info Ratio': f"{active.mean() * 252 / (active.std() * np.sqrt(252)):.3f}" if active.std() > 0 else 'N/A',
        'Win Rate': f"{(returns > 0).mean():.1%}",
        'Best Day': f"{returns.max():.2%}",
        'Worst Day': f"{returns.min():.2%}",
    }


metrics_port = compute_all_metrics(port, bench)
metrics_bench = compute_all_metrics(bench, bench)

print(f"\n{'Metric':<22} {'Portfolio':>14} {'Benchmark':>14}")
print("-" * 54)
for key in metrics_port:
    print(f"{key:<22} {metrics_port[key]:>14} {metrics_bench[key]:>14}")

# --- Visualization ---

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Risk Budgeting & Performance Attribution",
             fontsize=18, fontweight='bold', y=1.02)

# Panel 1: Capital vs risk allocation
ax = axes[0, 0]
x = np.arange(len(rb.names))
width = 0.35
ax.bar(x - width/2, data['weights'].values * 100, width, color=COLORS['portfolio'],
       label='Capital Weight', alpha=0.8)
ax.bar(x + width/2, pct_contrib * 100, width, color=COLORS['var'],
       label='Risk Contribution', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(rb.names, rotation=45, ha='right', fontsize=9)
ax.set_title('Capital Allocation vs. Risk Contribution', fontsize=14)
ax.set_ylabel('Percentage (%)')
ax.legend(fontsize=10)

# Panel 2: Risk parity vs original
ax = axes[0, 1]
ax.bar(x - width/2, data['weights'].values * 100, width, color=COLORS['portfolio'],
       label='Original Weights', alpha=0.8)
ax.bar(x + width/2, rp_weights * 100, width, color=COLORS['risk_budget'],
       label='Risk Parity Weights', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(rb.names, rotation=45, ha='right', fontsize=9)
ax.set_title('Original vs. Risk Parity Weights', fontsize=14)
ax.set_ylabel('Weight (%)')
ax.legend(fontsize=10)

# Panel 3: Risk parity risk contribution (should be equal)
ax = axes[1, 0]
ax.bar(x, rp_pct_contrib * 100, color=COLORS['risk_budget'], alpha=0.8)
ax.axhline(100 / len(rb.names), color='red', linestyle='--', linewidth=1.5,
           label=f'Target: {100/len(rb.names):.1f}%')
ax.set_xticks(x)
ax.set_xticklabels(rb.names, rotation=45, ha='right', fontsize=9)
ax.set_title('Risk Parity: Risk Contribution (Should Be Equal)', fontsize=14)
ax.set_ylabel('% of Total Risk')
ax.legend(fontsize=10)

# Panel 4: Backtest risk parity vs original
ax = axes[1, 1]
rp_returns = data['returns'].values @ rp_weights
cum_rp = pd.Series((1 + pd.Series(rp_returns, index=data['returns'].index)).cumprod().values,
                    index=data['returns'].index)
cum_orig = (1 + port).cumprod()
cum_bm = (1 + bench).cumprod()

ax.plot(cum_orig.index, cum_orig, color=COLORS['portfolio'], linewidth=1.5, label='Original Portfolio')
ax.plot(cum_rp.index, cum_rp, color=COLORS['risk_budget'], linewidth=1.5, label='Risk Parity')
ax.plot(cum_bm.index, cum_bm, color=COLORS['benchmark'], linewidth=1.5, linestyle='--', label='Benchmark (60/40)')
ax.set_title('Cumulative Returns: Original vs. Risk Parity vs. Benchmark', fontsize=14)
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


# ========================================
# Module Summary
# ========================================

print("\n" + "=" * 80)
print(" " * 18 + "MODULE 4.2 — SUMMARY")
print("=" * 80)
print("""
Key Concepts Covered:

  1. VALUE AT RISK (VaR)
     • Three methods: parametric (fast, assumes normality),
       historical (non-parametric, backward-looking),
       Monte Carlo (flexible, computationally expensive)
     • VaR is NOT subadditive — it can penalize diversification
     • Always backtest your VaR: breach rate should match confidence level
     • WHY QUANTS NEED IT: regulatory capital, margin, risk limits

  2. EXPECTED SHORTFALL (CVaR)
     • The average loss in the tail — what VaR refuses to tell you
     • A coherent risk measure (subadditive, rewards diversification)
     • Basel III/IV shifting from VaR to ES as primary measure
     • WHY QUANTS NEED IT: tail risk, coherent capital allocation

  3. MAXIMUM DRAWDOWN
     • The metric investors actually feel — peak-to-trough decline
     • Duration matters as much as depth (career risk, redemption risk)
     • Calmar ratio = return / max drawdown
     • WHY QUANTS NEED IT: investor communication, strategy evaluation

  4. PERFORMANCE ATTRIBUTION & RISK BUDGETING
     • Decompose returns into allocation, selection, and interaction
     • Marginal risk contribution reveals who is really driving portfolio risk
     • Risk parity: equalize risk contribution, not capital allocation
     • WHY QUANTS NEED IT: understand what worked, allocate risk wisely

  Every risk metric is a lens. VaR sees the boundary. ES sees the tail.
  Drawdown sees the path. Attribution sees the sources.
  A complete risk framework uses all of them.
""")
print("=" * 80)
print("\n📚 Next: Module 4.3 — Advanced Portfolio Techniques\n")